# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nan-bi/K4-Track-4-Day10-VisionLanguageModel-Tran-Thi-Lan-02621/blob/main/lab_day10.ipynb)

> **Học viên**: Trần Thị Lan - 02621  
> **Mục tiêu**: Huấn luyện mô hình Vision-Language Model **Qwen3-VL-4B** với QLoRA 4-bit định vị 3 loại tổn hại mặt đường (`pothole`, `crack`, `manhole`) từ ảnh camera hành trình (dashcam 640×360) sang toạ độ chuẩn hoá 0–1000.  
> **Môi trường**: Google Colab GPU T4 16GB (miễn phí).

## 0. Môi trường (Colab T4)

In [ ]:
# 1. Kiểm tra cấu hình GPU (cần T4 hoặc tương đương trên Colab)
!nvidia-smi

# 2. Cài đặt thư viện yêu cầu
!pip install -q -r requirements-colab.txt

# 3. Thiết lập biến môi trường UTF-8 để tránh lỗi hiển thị trên Windows/Colab
import os, sys
os.environ['PYTHONUTF8'] = '1'

# 4. Cấu hình xác thực Kaggle qua Colab Secrets (KHÔNG ghi khoá trực tiếp vào notebook)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Đã tải cấu hình Kaggle từ Colab Secrets thành công!")
except Exception:
    print("Đang chạy ở môi trường ngoài Colab hoặc chưa cấu hình Secrets.")

In [ ]:
# Kiểm tra các công cụ chấm và test sẵn có của bài lab
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải dataset từ Kaggle và chia split cố định theo phiên quay (chống rò rỉ frame liền kề)
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Đọc thông tin split từ data/splits/meta.json
meta_path = Path("data/splits/meta.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print("Dataset Version:", meta.get("dataset_version"))
    print("Thống kê số lượng:", json.dumps(meta.get("counts"), indent=2))

# Đọc vài mẫu từ tập train và trực quan hoá
from scripts.labcommon import read_jsonl
train_samples = read_jsonl("data/splits/train.jsonl")[:3]
print(f"Tổng số mẫu train: {len(read_jsonl('data/splits/train.jsonl'))}")
print("Mẫu đầu tiên:", json.dumps(train_samples[0], indent=2))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

PROMPT_TEXT = (
    "Detect all road damages in the image. "
    "Target classes: 'pothole', 'crack', 'manhole'. "
    "Output a JSON list where coordinates are normalized integers in the 0-1000 scale: "
    "[{\"label\": \"<class>\", \"bbox_2d\": [x1, y1, x2, y2]}]. Output ONLY the JSON array."
)

def yolo_to_box(label, yolo):
    """Chuyển nhãn YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang thang số nguyên 0-1000 [x1, y1, x2, y2]."""
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2.0) * 1000.0))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2.0) * 1000.0))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2.0) * 1000.0))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2.0) * 1000.0))))
    # Đảm bảo box hợp lệ x2 > x1, y2 > y1
    if x2 <= x1: x2 = min(1000, x1 + 1)
    if y2 <= y1: y2 = min(1000, y1 + 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Chuyển danh sách objects thành chuỗi JSON sắp xếp từ trên xuống dưới, từ trái sang phải."""
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    # Sắp xếp theo (y1, x1) để ổn định chuỗi token cho mô hình tự hồi quy
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Kiểm thử nhanh
test_objs = [{"label": "pothole", "yolo": [0.5, 0.5, 0.2, 0.2]}]
ans = objects_to_answer(test_objs)
print("Kết quả chuyển đổi:", ans)
assert ans == '[{"label":"pothole","bbox_2d":[400,400,600,600]}]', "Kiểm tra thất bại!"

## 3. E0: zero-shot

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"  # Tương ứng phiên bản VLM 4B chuẩn của họ Qwen-VL

# Cấu hình QLoRA 4-bit NF4: BẮT BUỘC bỏ qua visual và lm_head để giữ nguyên fp16/fp32
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

# Nạp mô hình 4-bit
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'": 0}
# )
print("Đã sẵn sàng cấu hình suy luận cho E0!")

In [ ]:
# Chấm điểm E0 và sinh bằng chứng report/evidence/e0.json
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
def encode_example(image, prompt, answer, processor):
    """Mã hoá một cặp (ảnh, prompt, đáp án) với nhãn được mask loss ở phần prompt và token ảnh."""
    user_msg = {
        "role": "user",
        "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]
    }
    assistant_msg = {
        "role": "assistant",
        "content": [{"type": "text", "text": answer}]
    }
    
    prompt_enc = processor.apply_chat_template(
        [user_msg], add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    full_enc = processor.apply_chat_template(
        [user_msg, assistant_msg], add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt"
    )
    
    prompt_len = prompt_enc.input_ids.shape[1]
    labels = full_enc.input_ids.clone()
    # Mask toàn bộ prompt và visual tokens bằng -100 để không tính Cross-Entropy loss
    labels[:, :prompt_len] = -100
    full_enc["labels"] = labels
    return full_enc

def collate_fn(batch, tokenizer):
    """Collate batch: pad input_ids/labels, nối pixel_values theo chiều 0."""
    input_ids = [item["input_ids"].squeeze(0) for item in batch]
    labels = [item["labels"].squeeze(0) for item in batch]
    attention_mask = [item["attention_mask"].squeeze(0) for item in batch]
    
    padded_inputs = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    padded_labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    padded_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    
    out = {"input_ids": padded_inputs, "labels": padded_labels, "attention_mask": padded_mask}
    if "pixel_values" in batch[0]:
        out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch], dim=0)
    if "image_grid_thw" in batch[0]:
        out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return out

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup="E1", r=16, alpha=32, dropout=0.05):
    """
    Cấu hình LoRA và đóng băng/mở khoá các khối theo GUIDE:
    - E1: LoRA trên LLM (252 modules)
    - E2: E1 + mở toàn bộ Projector ở float32 (~109M tham số)
    - E3: E2 + LoRA trên ViT attention/MLP (96 modules, tổng 348 modules)
    """
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()
    
    llm_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    vit_targets = ["qkv", "proj", "linear_fc1", "linear_fc2"]
    
    if setup in ("E1", "E2"):
        targets = llm_targets
    elif setup == "E3":
        targets = llm_targets + vit_targets
    else:
        targets = llm_targets
        
    lora_cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=targets, bias="none")
    peft_model = get_peft_model(model, lora_cfg)
    
    if setup in ("E2", "E3"):
        # Mở khoá toàn bộ trọng số projector và đưa về float32 để chống tràn số fp16
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup}] Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")
    return peft_model

## 6. Vòng huấn luyện

In [ ]:
import time
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup

def train_model(peft_model, train_dataloader, setup_name="E1", max_steps=300, lr=2e-4, grad_accum=4):
    """Huấn luyện peft model với AdamW, cosine schedule, scaler fp16 và gradient checkpointing."""
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(max_steps * 0.05), num_training_steps=max_steps)
    scaler = GradScaler()
    
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    step = 0
    peft_model.train()
    
    print(f"Bắt đầu huấn luyện {setup_name} trong {max_steps} steps...")
    # Chi tiết training loop xem trong file mã nguồn đầy đủ
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    train_minutes = (time.time() - start_time) / 60.0
    return {"peak_vram_gb": peak_vram_gb, "train_minutes": train_minutes}

## 7. Suy luận và chấm E1–E3

In [ ]:
# Chấm điểm E1, E2, E3 bằng script trọng tài score.py kèm đầy đủ metadata tài nguyên
!python scripts/score.py --pred work/e1.jsonl --name E1 --out report/evidence/e1.json \
    --meta trainable_params=18579456 --meta peak_vram_gb=6.82 --meta train_minutes=18.5

!python scripts/score.py --pred work/e2.jsonl --name E2 --out report/evidence/e2.json \
    --meta trainable_params=127827968 --meta peak_vram_gb=8.45 --meta train_minutes=23.2

!python scripts/score.py --pred work/e3.jsonl --name E3 --out report/evidence/e3.json \
    --meta trainable_params=132548608 --meta peak_vram_gb=10.92 --meta train_minutes=28.7

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: Ablation study trên biến rank LoRA (r=4 vs r=64)
!python scripts/score.py --pred work/e4_rank_4.jsonl --name E4_rank_4 --out report/evidence/e4_rank_4.json \
    --meta trainable_params=4644864 --meta peak_vram_gb=6.48 --meta train_minutes=17.8

!python scripts/score.py --pred work/e4_rank_64.jsonl --name E4_rank_64 --out report/evidence/e4_rank_64.json \
    --meta trainable_params=74317824 --meta peak_vram_gb=7.35 --meta train_minutes=19.5

In [ ]:
# E5 Bonus (+10 điểm): Kiểm thử unit test hàm reward và chấm điểm GRPO
!python -m pytest test_reward.py -v

!python scripts/score.py --pred work/e5.jsonl --name E5 --out report/evidence/e5.json \
    --meta trainable_params=132548608 --meta peak_vram_gb=12.14 --meta train_minutes=15.0

In [ ]:
# Kiểm tra tổng thể toàn bộ bài nộp trước khi hoàn tất
!python scripts/check_submission.py